# Generasi Data untuk Pelatihan SEM

Notebook ini menghasilkan dua dataset dummy untuk pelatihan **Structural Equation Modeling (SEM)** di R:

1. **`sem_ecommerce.csv`** — Untuk **CB-SEM** (*Covariance-Based SEM*, e.g., `lavaan`)
2. **`sem_innovation.csv`** — Untuk **PLS-SEM** (*Partial Least Squares SEM*, e.g., `seminr`)

Kedua dataset dirancang dengan dasar teori yang kuat agar hasilnya **interpretable secara ekonomi dan manajemen**. Semua indikator menggunakan skala Likert 1–7.

In [1]:
library(MASS)

# Helper: konversi skor laten ke skala Likert 1-7 (indikator reflektif)
to_likert7 <- function(latent, loading, center = 4) {
  x <- center + loading * latent + rnorm(length(latent), 0, sqrt(1 - loading^2))
  as.integer(pmin(pmax(round(x), 1), 7))
}

# Helper: clamp ke Likert 1-7 (untuk indikator formatif)
clamp17 <- function(x) as.integer(pmin(pmax(round(x), 1), 7))

## Dataset 1: CB-SEM — Kepuasan Pelanggan E-Commerce

### Dasar Teori

Model ini mengadaptasi kerangka **American Customer Satisfaction Index / ACSI** (Fornell et al., 1996) dan teori **SERVQUAL** (Parasuraman et al., 1988). Model klasik ini telah divalidasi secara luas dalam riset pemasaran dan perilaku konsumen.

| Konstruk | Kode | Tipe | Jumlah Item | Sumber Teori |
|---|---|---|---|---|
| Kualitas Layanan | SQ | Exogenous, Reflektif | 4 | SERVQUAL (Parasuraman et al., 1988) |
| Persepsi Nilai | PV | Exogenous, Reflektif | 3 | Perceived Value (Zeithaml, 1988) |
| Kepercayaan | TR | Exogenous, Reflektif | 4 | Trust Theory (Morgan & Hunt, 1994) |
| Kepuasan Pelanggan | CS | Endogenous, Reflektif | 4 | ACSI (Fornell et al., 1996) |
| Loyalitas Pelanggan | CL | Endogenous, Reflektif | 3 | Customer Loyalty (Oliver, 1999) |
| Niat Beli Ulang | RI | Endogenous, Reflektif | 3 | Behavioral Intentions (Zeithaml et al., 1996) |

### Model Struktural (Path Coefficients)

```
SQ ──(β=0.35)──→ CS ──(β=0.50)──→ CL ──(β=0.40)──→ RI
PV ──(β=0.25)──→ CS ──(β=0.30)──────────────────────→ RI
TR ──(β=0.30)──→ CS
TR ──(β=0.15)──────→ CL
```

### Contoh Spesifikasi `lavaan`

```r
library(lavaan)
model <- '
  # Measurement Model
  SQ =~ SQ1 + SQ2 + SQ3 + SQ4
  PV =~ PV1 + PV2 + PV3
  TR =~ TR1 + TR2 + TR3 + TR4
  CS =~ CS1 + CS2 + CS3 + CS4
  CL =~ CL1 + CL2 + CL3
  RI =~ RI1 + RI2 + RI3

  # Structural Model
  CS ~ SQ + PV + TR
  CL ~ CS + TR
  RI ~ CS + CL
'
fit <- sem(model, data = read.csv("sem_ecommerce.csv"))
summary(fit, fit.measures = TRUE, standardized = TRUE)
```

In [2]:
set.seed(2026)
n <- 300

# --- Model Struktural: Variabel Laten ---
# Variabel eksogen berkorelasi (SQ, PV, TR)
Sigma_exo <- matrix(c(
  1.00, 0.35, 0.40,
  0.35, 1.00, 0.30,
  0.40, 0.30, 1.00
), 3, 3)

exo <- mvrnorm(n, mu = rep(0, 3), Sigma = Sigma_exo)
SQ <- exo[, 1]  # Kualitas Layanan
PV <- exo[, 2]  # Persepsi Nilai
TR <- exo[, 3]  # Kepercayaan

# Variabel endogen
CS <- 0.35 * SQ + 0.25 * PV + 0.30 * TR + rnorm(n, 0, sqrt(0.40))  # Kepuasan
CL <- 0.50 * CS + 0.15 * TR + rnorm(n, 0, sqrt(0.45))              # Loyalitas
RI <- 0.30 * CS + 0.40 * CL + rnorm(n, 0, sqrt(0.35))              # Niat Beli Ulang

# --- Model Pengukuran: Indikator Reflektif (Likert 1-7) ---
data_cbsem <- data.frame(
  id = 1:n,
  # Demografi
  gender    = sample(c("Laki-laki", "Perempuan"), n, replace = TRUE),
  age       = sample(18:55, n, replace = TRUE),
  education = sample(c("SMA", "D3", "S1", "S2"), n, replace = TRUE,
                     prob = c(0.15, 0.20, 0.45, 0.20)),
  income    = sample(c("< 3 juta", "3-6 juta", "6-10 juta", "> 10 juta"),
                     n, replace = TRUE, prob = c(0.20, 0.35, 0.30, 0.15)),
  # Kualitas Layanan (loading: 0.74–0.82)
  SQ1 = to_likert7(SQ, 0.82), SQ2 = to_likert7(SQ, 0.79),
  SQ3 = to_likert7(SQ, 0.76), SQ4 = to_likert7(SQ, 0.74),
  # Persepsi Nilai (loading: 0.75–0.83)
  PV1 = to_likert7(PV, 0.83), PV2 = to_likert7(PV, 0.78),
  PV3 = to_likert7(PV, 0.75),
  # Kepercayaan (loading: 0.73–0.81)
  TR1 = to_likert7(TR, 0.81), TR2 = to_likert7(TR, 0.78),
  TR3 = to_likert7(TR, 0.76), TR4 = to_likert7(TR, 0.73),
  # Kepuasan Pelanggan (loading: 0.74–0.84)
  CS1 = to_likert7(CS, 0.84), CS2 = to_likert7(CS, 0.80),
  CS3 = to_likert7(CS, 0.77), CS4 = to_likert7(CS, 0.74),
  # Loyalitas Pelanggan (loading: 0.76–0.83)
  CL1 = to_likert7(CL, 0.83), CL2 = to_likert7(CL, 0.79),
  CL3 = to_likert7(CL, 0.76),
  # Niat Beli Ulang (loading: 0.77–0.85)
  RI1 = to_likert7(RI, 0.85), RI2 = to_likert7(RI, 0.81),
  RI3 = to_likert7(RI, 0.77)
)

write.csv(data_cbsem, "sem_ecommerce.csv", row.names = FALSE)
tibble::glimpse(data_cbsem)

Rows: 300
Columns: 26
$ id        <int> 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 1…
$ gender    <chr> "Perempuan", "Laki-laki", "Perempuan", "Perempuan", "Laki-la…
$ age       <int> 50, 25, 48, 21, 43, 25, 39, 30, 54, 55, 46, 22, 46, 20, 32, …
$ education <chr> "S2", "S1", "S2", "SMA", "D3", "S1", "S2", "SMA", "S2", "S1"…
$ income    <chr> "6-10 juta", "> 10 juta", "< 3 juta", "3-6 juta", "6-10 juta…
$ SQ1       <int> 5, 2, 4, 4, 3, 1, 3, 3, 4, 4, 3, 3, 4, 4, 1, 5, 5, 4, 3, 4, …
$ SQ2       <int> 4, 3, 4, 5, 3, 2, 3, 3, 4, 4, 4, 4, 5, 4, 2, 4, 5, 4, 4, 6, …
$ SQ3       <int> 5, 4, 4, 3, 4, 2, 3, 4, 3, 3, 4, 4, 5, 4, 2, 3, 5, 4, 3, 5, …
$ SQ4       <int> 5, 3, 5, 5, 4, 2, 3, 3, 4, 4, 5, 3, 5, 4, 3, 5, 3, 3, 3, 4, …
$ PV1       <int> 4, 3, 4, 3, 4, 4, 4, 3, 5, 4, 6, 4, 4, 3, 3, 6, 6, 5, 4, 5, …
$ PV2       <int> 5, 5, 5, 3, 4, 3, 6, 3, 4, 3, 5, 4, 3, 2, 4, 6, 5, 4, 3, 3, …
$ PV3       <int> 5, 3, 4, 4, 5, 2, 4, 2, 3, 3, 5, 3, 3, 2, 3, 7, 5, 5, 4, 5, …
$ TR1       <int> 

## Dataset 2: PLS-SEM — Inovasi & Kinerja UMKM

### Dasar Teori

Model ini mengombinasikan tiga teori utama dalam manajemen strategis:

1. **Resource-Based View / RBV** (Barney, 1991) — Kapabilitas inovasi sebagai sumber daya strategis yang menghasilkan keunggulan bersaing berkelanjutan
2. **Market Orientation** (Narver & Slater, 1990) — Orientasi pasar mendorong penciptaan nilai superior bagi pelanggan
3. **Entrepreneurial Orientation** (Lumpkin & Dess, 1996) — Proaktivitas, pengambilan risiko, dan inovasi sebagai penggerak kinerja

Dataset ini secara khusus dirancang untuk mendemonstrasikan **konstruk formatif (Mode B)** pada PLS-SEM — di mana indikator **membentuk** konstruk, bukan sebaliknya.

| Konstruk | Kode | Tipe | Mode | Item | Indikator |
|---|---|---|---|---|---|
| Kapabilitas Inovasi | IC | Exogenous | **Formatif (B)** | 4 | Investasi R&D, Kreativitas, Adopsi Teknologi, Kolaborasi |
| Orientasi Pasar | MO | Exogenous | **Formatif (B)** | 3 | Orientasi Pelanggan, Pesaing, Koordinasi |
| Orientasi Kewirausahaan | EO | Exogenous | **Formatif (B)** | 3 | Proaktivitas, Risiko, Inovasi Produk |
| Keunggulan Bersaing | CA | Endogenous | Reflektif (A) | 4 | Diferensiasi, Efisiensi, Respons Pasar, Reputasi |
| Kinerja Perusahaan | FP | Endogenous | Reflektif (A) | 4 | Profitabilitas, Pertumbuhan, Pangsa Pasar, ROI |

### Model Struktural (Path Coefficients)

```
IC ──(β=0.35)──→ CA ──(β=0.50)──→ FP
MO ──(β=0.30)──→ CA
EO ──(β=0.25)──→ CA
IC ──(β=0.20)──────────────────→ FP  (direct effect)
```

### Contoh Spesifikasi `seminr`

```r
library(seminr)

mm <- constructs(
  composite("IC", multi_items("IC", 1:4), weights = mode_B),
  composite("MO", multi_items("MO", 1:3), weights = mode_B),
  composite("EO", multi_items("EO", 1:3), weights = mode_B),
  composite("CA", multi_items("CA", 1:4), weights = mode_A),
  composite("FP", multi_items("FP", 1:4), weights = mode_A)
)

sm <- relationships(
  paths(from = c("IC", "MO", "EO"), to = "CA"),
  paths(from = c("CA", "IC"), to = "FP")
)

pls_model <- estimate_pls(
  data = read.csv("sem_innovation.csv"),
  measurement_model = mm,
  structural_model = sm
)
summary(pls_model)
```

In [3]:
set.seed(2026)
n <- 200

# --- Indikator Formatif (dihasilkan secara independen) ---
# Kapabilitas Inovasi (IC) - FORMATIF
IC1 <- clamp17(rnorm(n, 4.5, 1.2))  # Investasi Riset & Pengembangan
IC2 <- clamp17(rnorm(n, 4.8, 1.1))  # Kreativitas & Ide Baru Karyawan
IC3 <- clamp17(rnorm(n, 5.0, 1.0))  # Adopsi Teknologi Baru
IC4 <- clamp17(rnorm(n, 4.3, 1.3))  # Kolaborasi dengan Pihak Eksternal

# Orientasi Pasar (MO) - FORMATIF
MO1 <- clamp17(rnorm(n, 5.0, 1.1))  # Orientasi Pelanggan
MO2 <- clamp17(rnorm(n, 4.5, 1.2))  # Orientasi Pesaing
MO3 <- clamp17(rnorm(n, 4.7, 1.1))  # Koordinasi Antar-fungsi

# Orientasi Kewirausahaan (EO) - FORMATIF
EO1 <- clamp17(rnorm(n, 4.6, 1.2))  # Proaktivitas
EO2 <- clamp17(rnorm(n, 4.4, 1.3))  # Pengambilan Risiko
EO3 <- clamp17(rnorm(n, 4.9, 1.1))  # Inovasi Produk/Jasa

# --- Komposit Konstruk Formatif ---
IC <- 0.30 * scale(IC1)[, 1] + 0.25 * scale(IC2)[, 1] +
      0.25 * scale(IC3)[, 1] + 0.20 * scale(IC4)[, 1]
MO <- 0.40 * scale(MO1)[, 1] + 0.30 * scale(MO2)[, 1] +
      0.30 * scale(MO3)[, 1]
EO <- 0.35 * scale(EO1)[, 1] + 0.35 * scale(EO2)[, 1] +
      0.30 * scale(EO3)[, 1]

# --- Model Struktural ---
CA_lat <- 0.35 * IC + 0.30 * MO + 0.25 * EO + rnorm(n, 0, 0.45)  # Keunggulan Bersaing
FP_lat <- 0.50 * CA_lat + 0.20 * IC + rnorm(n, 0, 0.40)           # Kinerja Perusahaan

# --- Indikator Reflektif untuk Konstruk Endogen ---
data_plssem <- data.frame(
  id = 1:n,
  # Kapabilitas Inovasi (formatif)
  IC1, IC2, IC3, IC4,
  # Orientasi Pasar (formatif)
  MO1, MO2, MO3,
  # Orientasi Kewirausahaan (formatif)
  EO1, EO2, EO3,
  # Keunggulan Bersaing (reflektif, loading: 0.74–0.82)
  CA1 = to_likert7(CA_lat, 0.82), CA2 = to_likert7(CA_lat, 0.79),
  CA3 = to_likert7(CA_lat, 0.77), CA4 = to_likert7(CA_lat, 0.74),
  # Kinerja Perusahaan (reflektif, loading: 0.75–0.84)
  FP1 = to_likert7(FP_lat, 0.84), FP2 = to_likert7(FP_lat, 0.81),
  FP3 = to_likert7(FP_lat, 0.78), FP4 = to_likert7(FP_lat, 0.75),
  # Variabel kontrol
  firm_size = sample(c("Mikro", "Kecil", "Menengah"), n, replace = TRUE,
                     prob = c(0.30, 0.45, 0.25)),
  sector    = sample(c("Manufaktur", "Jasa", "Perdagangan", "Teknologi"),
                     n, replace = TRUE),
  firm_age  = sample(1:25, n, replace = TRUE)
)

write.csv(data_plssem, "sem_innovation.csv", row.names = FALSE)
tibble::glimpse(data_plssem)

Rows: 200
Columns: 22
$ id        <int> 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 1…
$ IC1       <int> 5, 3, 5, 4, 4, 1, 4, 3, 5, 4, 4, 4, 4, 4, 1, 6, 5, 5, 4, 5, …
$ IC2       <int> 4, 5, 5, 3, 4, 7, 4, 4, 5, 7, 4, 6, 4, 4, 4, 6, 5, 6, 6, 7, …
$ IC3       <int> 6, 6, 4, 5, 4, 3, 7, 7, 5, 4, 6, 4, 6, 6, 5, 5, 6, 6, 5, 5, …
$ IC4       <int> 6, 4, 4, 5, 4, 4, 4, 4, 3, 5, 5, 3, 7, 4, 4, 2, 4, 3, 5, 5, …
$ MO1       <int> 3, 4, 4, 4, 6, 5, 6, 5, 5, 7, 4, 5, 7, 5, 7, 3, 5, 4, 3, 4, …
$ MO2       <int> 7, 6, 6, 4, 6, 4, 5, 5, 4, 5, 6, 5, 3, 3, 4, 3, 5, 4, 4, 2, …
$ MO3       <int> 5, 6, 5, 4, 6, 3, 6, 5, 2, 4, 6, 2, 6, 6, 6, 5, 4, 3, 5, 4, …
$ EO1       <int> 7, 2, 3, 6, 6, 4, 3, 3, 6, 2, 4, 5, 4, 5, 7, 4, 5, 4, 5, 4, …
$ EO2       <int> 3, 6, 4, 6, 4, 6, 4, 4, 5, 3, 4, 4, 6, 3, 6, 4, 4, 6, 5, 6, …
$ EO3       <int> 3, 4, 5, 4, 6, 5, 4, 4, 4, 4, 6, 4, 6, 5, 3, 4, 3, 7, 5, 5, …
$ CA1       <int> 4, 4, 4, 4, 3, 4, 4, 3, 4, 4, 4, 4, 4, 4, 5, 4, 3, 5, 3, 4, …
$ CA2       <int> 

## Dataset 3: WVS Wave 7 — Data Regresi (Variabel Bahasa Indonesia)

Rename semua variabel dari Bahasa Inggris ke Bahasa Indonesia, serta terjemahkan nilai-nilai kategorikal. Menghasilkan dua file:

- `wvs1.csv` — Singapura, Kanada, Selandia Baru
- `wvs2.csv` — Singapura, Thailand, Indonesia

In [5]:
library(dplyr)
library(readr)

# --- Baca data asli ---
wvs1 <- read_csv("wvs-wave7-sg-ca-nz.csv", show_col_types = FALSE)
wvs2 <- read_csv("wvs-wave7-sg-th-id.csv", show_col_types = FALSE)

# --- Fungsi rename & translate ---
translate_wvs <- function(df) {
  df |>
    rename(
      negara               = country,
      id_responden         = ID,
      pentingnya_keluarga  = family_importance,
      pentingnya_teman     = friends_importance,
      pentingnya_waktu_luang = leisure_importance,
      pentingnya_pekerjaan = work_importance,
      kebebasan_memilih    = freedom,
      kepuasan_hidup       = life_satisfaction,
      kepuasan_finansial   = financial_satisfaction,
      religiusitas         = religiousity,
      skala_politik        = political_scale,
      jenis_kelamin        = sex,
      tahun_lahir          = birthyear,
      usia                 = age,
      status_pernikahan    = marital_status,
      status_pekerjaan     = employment
    ) |>
    mutate(
      negara = recode(negara,
        "CAN" = "Kanada", "SGP" = "Singapura", "NZL" = "Selandia Baru",
        "IDN" = "Indonesia", "THA" = "Thailand"
      ),
      jenis_kelamin = recode(jenis_kelamin,
        "Male" = "Laki-laki", "Female" = "Perempuan"
      ),
      religiusitas = recode(religiusitas,
        "A religious person"     = "Religius",
        "Not a religious person" = "Tidak religius",
        "An atheist"             = "Ateis",
        "Don't know"             = "Tidak tahu"
      ),
      status_pernikahan = recode(status_pernikahan,
        "Married"                      = "Menikah",
        "Living together as married"   = "Kohabitasi",
        "Divorced"                     = "Bercerai",
        "Separated"                    = "Pisah",
        "Widowed"                      = "Janda/Duda",
        "Single"                       = "Lajang"
      ),
      status_pekerjaan = recode(status_pekerjaan,
        "Full time"    = "Penuh waktu",
        "Part time"    = "Paruh waktu",
        "Self employed" = "Wiraswasta",
        "Retired/pensioned" = "Pensiunan",
        "Homemaker not otherwise employed" = "Ibu rumah tangga",
        "Homemaker"    = "Ibu rumah tangga",
        "Student"      = "Pelajar/Mahasiswa",
        "Unemployed"   = "Tidak bekerja",
        "Other"        = "Lainnya"
      )
    )
}

# --- Terapkan dan simpan ---
wvs1_id <- translate_wvs(wvs1)
wvs2_id <- translate_wvs(wvs2)

write_csv(wvs1_id, "wvs1.csv")
write_csv(wvs2_id, "wvs2.csv")

tibble::glimpse(wvs1_id)
tibble::glimpse(wvs2_id)

Rows: 7,087
Columns: 16
$ negara                 <chr> "Kanada", "Kanada", "Kanada", "Kanada", "Kanada…
$ id_responden           <dbl> 124070003, 124070004, 124070005, 124070006, 124…
$ pentingnya_keluarga    <dbl> 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2,…
$ pentingnya_teman       <dbl> 1, 1, 2, 2, 2, 2, 1, 2, 1, 3, 1, 2, 1, 1, 1, 3,…
$ pentingnya_waktu_luang <dbl> 1, 2, 2, 2, 2, 2, 1, 2, 1, 2, 1, 1, 2, 2, 1, 2,…
$ pentingnya_pekerjaan   <dbl> 1, 1, 2, 2, 2, 1, 1, 2, 1, 2, 1, 2, 4, 1, 3, 3,…
$ kebebasan_memilih      <dbl> 7, 5, 8, 4, 7, 6, 9, 7, 8, 6, 8, 10, 8, 9, 7, 6…
$ kepuasan_hidup         <dbl> 5, 8, 9, 7, 1, 7, 9, 6, 7, 6, 7, 10, 8, 10, 6, …
$ kepuasan_finansial     <dbl> 8, 2, 8, 8, 9, 8, 9, 5, 6, 9, 5, 7, 9, 9, 7, 8,…
$ religiusitas           <chr> "Religius", "Religius", "Ateis", "Religius", "T…
$ skala_politik          <dbl> 10, 5, 2, 8, 6, 6, 7, 5, 6, 4, 5, 4, 1, 7, 2, 8…
$ jenis_kelamin          <chr> "Perempuan", "Laki-laki", "Perempuan", "Laki-la…
$ tahun_lahir   